# Mass Shootings per Year — Bar Chart with Legislation Overlay

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df = pd.read_csv('Mother Jones - Mass Shootings Database, 1982 - 2026 - Sheet1.csv')
df['fatalities'] = pd.to_numeric(df['fatalities'], errors='coerce')
# Dataset mixes 4-digit years (1982–1999) and 2-digit years (2000s onward)
df['date_parsed'] = pd.to_datetime(df['date'], format='mixed', dayfirst=False, errors='coerce')
df = df.dropna(subset=['date_parsed', 'fatalities']).copy()
df['year'] = df['date_parsed'].dt.year
print(f"{len(df)} shootings, {df['year'].min()}–{df['year'].max()}")

159 shootings, 1982–2026


In [15]:
DROP_COLS = [
    'summary', 'mental_health_details', 'mental_health_sources',
    'sources', 'sources_additional_age', 'where_obtained', 'weapon_details',
]
df.drop(columns=DROP_COLS, inplace=True)
df.to_csv('shootings_clean.csv', index=False)
print(f"Columns kept: {list(df.columns)}")
print(f"Saved → shootings_clean.csv  ({len(df)} rows, {len(df.columns)} columns)")

Columns kept: ['case', 'location', 'date', 'fatalities', 'injured', 'total_victims', 'location.1', 'age_of_shooter', 'prior_signs_mental_health_issues', 'weapons_obtained_legally', 'weapon_type', 'race', 'gender', 'latitude', 'longitude', 'type', 'year', 'date_parsed']
Saved → shootings_clean.csv  (159 rows, 18 columns)


In [ ]:
per_year = df.groupby('year').size().reset_index(name='count')
all_years = pd.DataFrame({'year': range(df['year'].min(), df['year'].max() + 1)})
per_year = all_years.merge(per_year, on='year', how='left').fillna(0)
per_year['count'] = per_year['count'].astype(int)

LEGISLATION = [
    (1993, 'Brady Bill\n(background checks)', 'enacted', '#1a6b3c'),
    (1994, 'Assault Weapons\nBan enacted',    'enacted', '#1a3a5c'),
    (2004, 'Assault Weapons\nBan expires',    'expired', '#c0392b'),
    (2022, 'Bipartisan Safer\nCommunities Act','enacted', '#1a3a5c'),
]

plt.rcParams["font.family"] = "DejaVu Serif"
plt.rcParams["text.color"] = "#333333"
plt.rcParams["axes.labelcolor"] = "#333333"
plt.rcParams["xtick.color"] = "#333333"
plt.rcParams["ytick.color"] = "#333333"

fig, ax = plt.subplots(figsize=(18, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

cmap = plt.cm.jet
norm = plt.Normalize(0, per_year['count'].max())
bar_colors = [cmap(norm(c)) for c in per_year['count']]

ax.bar(per_year['year'], per_year['count'], color=bar_colors, width=0.75, zorder=3)

y_max = per_year['count'].max()
for year, label, kind, color in LEGISLATION:
    ls = '--' if kind == 'expired' else '-'
    ax.axvline(year, color=color, linewidth=1.4, linestyle=ls, alpha=0.85, zorder=4)
    label_y = y_max * 0.94 if year in (1993, 2004) else y_max * 0.76
    ax.text(year + 0.25, label_y, label,
            color=color, fontsize=8.5, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, lw=0.8, alpha=0.92))

ax.set_xlim(per_year['year'].min() - 0.75, per_year['year'].max() + 0.75)
ax.set_ylim(0, y_max + 3)
ax.set_xlabel('Year', fontsize=11, labelpad=8)
ax.set_ylabel('Shootings', fontsize=11, labelpad=8)
ax.set_xticks(range(per_year['year'].min(), per_year['year'].max() + 1, 2))
ax.tick_params(axis='x', rotation=45, labelsize=9.5)
ax.tick_params(axis='y', labelsize=9.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.yaxis.grid(True, color='#e8e8e8', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

enacted_patch = mpatches.Patch(color='#1a3a5c', label='Legislation enacted')
expired_patch = mpatches.Patch(color='#c0392b', label='Legislation expired')
ax.legend(handles=[enacted_patch, expired_patch],
          loc='upper left', fontsize=9, framealpha=0.9, edgecolor='#cccccc')

ax.set_title('Mass Shootings per Year, 1982–2026',
             fontsize=15, fontweight='bold', pad=14, loc='left', color='#1a1a1a')
ax.annotate(f'{len(df)} incidents · Source: Mother Jones Mass Shootings Database',
            xy=(0, 1.01), xycoords='axes fraction',
            fontsize=9, color='#777', va='bottom')

plt.tight_layout(pad=1.5)
plt.savefig('barchart_per_year.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved → barchart_per_year.png')

# Cumulative Casualties — Running Total Since 1982

In [17]:
import plotly.graph_objects as go

cumul = (
    df[['date_parsed', 'fatalities']]
    .sort_values('date_parsed')
    .assign(cumulative=lambda d: d['fatalities'].cumsum())
)
total = int(cumul['cumulative'].iloc[-1])

year_dots = (cumul.assign(year=cumul['date_parsed'].dt.year)
                  .groupby('year', as_index=False).last())

LEGISLATION = [
    (1993, 'Brady Bill (background checks)', 'enacted', '#1a6b3c'),
    (1994, 'Assault Weapons Ban enacted',    'enacted', '#1a3a5c'),
    (2004, 'Assault Weapons Ban expires',    'expired', '#c0392b'),
    (2022, 'Bipartisan Safer Communities Act','enacted', '#1a3a5c'),
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cumul['date_parsed'], y=cumul['cumulative'],
    fill='tozeroy', fillcolor='rgba(26,58,92,0.08)',
    line=dict(color='#1a3a5c', width=2.5),
    hovertemplate='%{x|%b %Y}<br>%{y:,} killed<extra></extra>',
    name='cumulative',
))

fig.add_trace(go.Scatter(
    x=year_dots['date_parsed'], y=year_dots['cumulative'],
    mode='markers',
    marker=dict(size=8, color='white', line=dict(color='#1a3a5c', width=1.8)),
    hovertemplate='<b>End of %{customdata}</b><br>%{y:,} cumulative killed<extra></extra>',
    customdata=year_dots['year'],
    name='year end',
))

label_y_offsets = {1993: 0.72, 1994: 0.55, 2004: 0.72, 2022: 0.55}
for year, label, kind, color in LEGISLATION:
    ts = f'{year}-07-01'
    fig.add_vline(x=ts,
                  line=dict(color=color, width=1.4,
                             dash='dash' if kind == 'expired' else 'solid'),
                  opacity=0.85)
    fig.add_annotation(
        x=ts, y=label_y_offsets[year], yref='paper',
        text=label, showarrow=False,
        font=dict(size=9, color=color, family='serif'),
        bgcolor='white', bordercolor=color, borderwidth=0.8,
        xanchor='left', xshift=6,
    )

fig.add_annotation(
    x=cumul['date_parsed'].iloc[-1], y=total,
    text=f'<b>{total:,} killed</b>',
    showarrow=True, arrowhead=2, arrowcolor='#888', arrowwidth=1,
    ax=-100, ay=-40,
    font=dict(size=12, color='#c0392b', family='serif'),
    bgcolor='white', bordercolor='#cccccc', borderwidth=0.8,
)

fig.update_layout(
    title=dict(text='Cumulative Lives Lost to Mass Shootings, 1982–2026',
               x=0, xanchor='left', font=dict(size=16, color='#1a1a1a', family='serif')),
    paper_bgcolor='white', plot_bgcolor='white',
    xaxis=dict(showgrid=False, color='#555', tickfont=dict(color='#555', family='serif'),
               linecolor='#cccccc'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.07)', color='#555',
               title='Cumulative fatalities',
               tickfont=dict(color='#555', family='serif'),
               zeroline=False),
    showlegend=False,
    margin=dict(l=60, r=20, t=60, b=40),
    annotations=[a for a in fig.layout.annotations] + [dict(
        text='Source: Mother Jones Mass Shootings Database',
        x=0, y=-0.1, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#777', family='serif'),
    )],
    hoverlabel=dict(bgcolor='white', bordercolor='#cccccc',
                    font=dict(color='#1a1a1a', size=11, family='serif')),
)

fig.write_html('cumulative_casualties.html')
fig.show()
print(f'Saved → cumulative_casualties.html  (total: {total:,} killed)')

Saved → cumulative_casualties.html  (total: 1,186 killed)


# Where They Happen — Choropleth by State

In [18]:
%pip install plotly nbformat -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import plotly.express as px

STATE_ABBREV = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Lousiana': 'LA',
    'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC', 'D.C.': 'DC',
}

geo = df.copy()
geo['state_name'] = geo['location'].str.split(',').str[-1].str.strip()
geo['state_code'] = geo['state_name'].map(STATE_ABBREV)

unmapped = geo[geo['state_code'].isna()]['state_name'].unique()
if len(unmapped):
    print('Unmapped (check manually):', unmapped)
else:
    print('All states mapped successfully.')

by_state = (geo.dropna(subset=['state_code'])
               .groupby(['state_code', 'state_name'], as_index=False)
               .agg(shootings=('case', 'count'), killed=('fatalities', 'sum')))
by_state['hover'] = (by_state['state_name'] + '<br>'
                     + by_state['shootings'].astype(str) + ' shootings<br>'
                     + by_state['killed'].astype(str) + ' killed')

fig = px.choropleth(
    by_state,
    locations='state_code',
    locationmode='USA-states',
    color='shootings',
    scope='usa',
    color_continuous_scale='Jet',
    custom_data=['hover'],
)

fig.update_traces(
    hovertemplate='%{customdata[0]}<extra></extra>',
    marker_line_color='white', marker_line_width=0.8,
)

fig.update_layout(
    title=dict(
        text='Mass Shootings by State, 1982–2026',
        x=0, xanchor='left',
        font=dict(size=16, color='#1a1a1a', family='serif'),
        pad=dict(l=10),
    ),
    geo=dict(
        scope='usa', bgcolor='white',
        landcolor='#f0f0f0', lakecolor='white',
        showlakes=True, showland=True,
        coastlinecolor='#cccccc', coastlinewidth=0.8,
        subunitcolor='#cccccc', subunitwidth=0.5,
    ),
    paper_bgcolor='white',
    coloraxis_colorbar=dict(
        title=dict(text='Shootings', font=dict(color='#555', family='serif')),
        tickfont=dict(color='#555', family='serif'), thickness=12, len=0.5,
    ),
    margin=dict(l=0, r=0, t=50, b=10),
    annotations=[dict(
        text=f'{len(df)} incidents · Source: Mother Jones Mass Shootings Database',
        x=0, y=-0.02, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#777', family='serif'),
    )],
    hoverlabel=dict(bgcolor='white', bordercolor='#cccccc',
                    font=dict(color='#1a1a1a', size=11, family='serif')),
)

fig.write_html('map_mass_shootings.html')
fig.show()
print(by_state.sort_values('shootings', ascending=False).to_string(index=False))